In [ ]:
import os
from pathlib import Path
import duckdb
import pandas

os.chdir(r"D:\Internship\Project\olist\olist-data-pipeline-1")

os.chdir(repo)
con = duckdb.connect("data/olist.duckdb")
con.execute(open("sql/01_load_raw.sql").read())
con.sql("SELECT schema_name, table_name, estimated_size FROM duckdb_tables()").show()

┌─────────────┬──────────────────────┬────────────────┐
│ schema_name │      table_name      │ estimated_size │
│   varchar   │       varchar        │     int64      │
├─────────────┼──────────────────────┼────────────────┤
│ dq          │ results              │             11 │
│ dq          │ summary              │             11 │
│ mart        │ dim_customer         │          99441 │
│ mart        │ dim_date             │           1096 │
│ mart        │ dim_product          │          32951 │
│ mart        │ dim_seller           │           3095 │
│ mart        │ fact_order_items     │         112650 │
│ raw         │ category_translation │             71 │
│ raw         │ customers            │          99441 │
│ raw         │ geolocation          │        1000163 │
│ raw         │ orders               │          99441 │
│ raw         │ order_items          │         112650 │
│ raw         │ payments             │         103886 │
│ raw         │ products             │          

In [7]:
con.execute(open("sql/02_staging.sql").read())
con.sql("SELECT order_status, count(*) n, avg(delivery_days) avg_days FROM stg.orders GROUP BY 1 ORDER BY n DESC").show()

┌──────────────┬───────┬────────────────────┐
│ order_status │   n   │      avg_days      │
│   varchar    │ int64 │       double       │
├──────────────┼───────┼────────────────────┤
│ delivered    │ 96478 │ 12.496848761272934 │
│ shipped      │  1107 │               NULL │
│ canceled     │   625 │ 20.333333333333332 │
│ unavailable  │   609 │               NULL │
│ invoiced     │   314 │               NULL │
│ processing   │   301 │               NULL │
│ created      │     5 │               NULL │
│ approved     │     2 │               NULL │
└──────────────┴───────┴────────────────────┘



In [8]:
con.execute(open("sql/03_marts.sql").read())
con.sql("""
SELECT d.year, d.month, round(sum(f.total_value)) revenue, count(DISTINCT f.order_id) orders
FROM mart.fact_order_items f JOIN mart.dim_date d USING (date_key)
WHERE f.order_status = 'delivered'
GROUP BY 1,2 ORDER BY 1,2
""").show()

┌───────┬───────┬───────────┬────────┐
│ year  │ month │  revenue  │ orders │
│ int64 │ int64 │  double   │ int64  │
├───────┼───────┼───────────┼────────┤
│  2016 │     9 │     143.0 │      1 │
│  2016 │    10 │   46491.0 │    265 │
│  2016 │    12 │      20.0 │      1 │
│  2017 │     1 │  127482.0 │    750 │
│  2017 │     2 │  271239.0 │   1653 │
│  2017 │     3 │  414331.0 │   2546 │
│  2017 │     4 │  390812.0 │   2303 │
│  2017 │     5 │  566851.0 │   3546 │
│  2017 │     6 │  490050.0 │   3135 │
│  2017 │     7 │  566299.0 │   3872 │
│  2017 │     8 │  645832.0 │   4193 │
│  2017 │     9 │  701077.0 │   4150 │
│  2017 │    10 │  751117.0 │   4478 │
│  2017 │    11 │ 1153364.0 │   7289 │
│  2017 │    12 │  843078.0 │   5513 │
│  2018 │     1 │ 1077887.0 │   7069 │
│  2018 │     2 │  966168.0 │   6555 │
│  2018 │     3 │ 1120598.0 │   7003 │
│  2018 │     4 │ 1132879.0 │   6798 │
│  2018 │     5 │ 1128775.0 │   6749 │
│  2018 │     6 │ 1011978.0 │   6099 │
│  2018 │     7 │ 1027807

In [10]:
for f in ["sql/02_staging.sql", "sql/03_marts.sql", "sql/04_dq_checks.sql"]:
    con.execute(open(f).read())
con.sql("SELECT * FROM dq.summary").show()

┌───────────────────────────┬─────────────┬─────────┐
│        check_name         │ failed_rows │ status  │
│          varchar          │    int64    │ varchar │
├───────────────────────────┼─────────────┼─────────┤
│ orders_pk_unique          │           0 │ PASS    │
│ fact_pk_unique            │           0 │ PASS    │
│ orders_purchase_not_null  │           0 │ PASS    │
│ fact_customer_fk          │           0 │ PASS    │
│ fact_product_fk           │           0 │ PASS    │
│ price_positive            │           0 │ PASS    │
│ review_score_1_to_5       │           0 │ PASS    │
│ delivered_before_purchase │           0 │ PASS    │
│ canceled_but_delivered    │           6 │ WARN    │
│ delivered_missing_date    │           8 │ WARN    │
│ unknown_category          │         610 │ WARN    │
└───────────────────────────┴─────────────┴─────────┘
  11 rows                                 3 columns

